# AML - TIME TO EVENT - WIHTOUT COVARIATES 
This script creates a time-to-event phenotype file for electronic health record–defined acute myeloid leukemia. The earliest electronic health record corresponding to acute myeloid leukemia is considered the event. Time-to-event was calculated both from birth and from the date of the earliest electronic health record. 

In [ ]:
# Load packages 
import pandas as pd 
import os 

In [ ]:
# Set environment for workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [ ]:
# Download necessary files from workspace bucket 
!gsutil cp {my_bucket}/data/covariates/observation_age_period.tsv .

## AML TIME TO EVENT - ELECTRONIC HEALTH RECORDS

In [ ]:
import pandas 
import os 

dataset_39009800_person_sql = """
SELECT DISTINCT c.concept_id 
FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
JOIN (
    SELECT CAST(cr.id as string) AS id       
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
    WHERE concept_id IN (140352) AND full_text LIKE '%_rank1]%'
) a 
ON (
    c.path LIKE CONCAT('%.', a.id, '.%') 
    OR c.path LIKE CONCAT('%.', a.id) 
    OR c.path LIKE CONCAT(a.id, '.%') 
    OR c.path = a.id
) 
WHERE is_standard = 1 AND is_selectable = 1

UNION DISTINCT 

SELECT DISTINCT concept_id
FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
WHERE concept_id IN (4304199) AND is_standard = 1
"""
dataset_39009800_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_39009800_person_df.head(5)

# =====================================================================================
# Access electronic health records data for acute myeloid leukaemia 
# =====================================================================================

concept_ids = tuple(dataset_39009800_person_df["concept_id"])

query = f"""                                
SELECT person_id, concept_id, entry_datetime, domain, visit_concept_id, visit_occurrence_id, age_at_event, 
FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_all_events`
WHERE concept_id IN {concept_ids}
"""

aml_phenotype_occurrences = pd.read_gbq(query, dialect="standard")

## AML ELECTRONIC HEALTH RECORDS 

In [ ]:
# =====================================================================================
# Access electronic health records concepts for acute myeloid leukaemia 
# =====================================================================================

concept_information = f"""
SELECT * 
FROM `{os.environ["WORKSPACE_CDR"]}.concept`
WHERE concept_id IN {concept_ids}
"""

concept_information = pandas.read_gbq(
    concept_information,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

concept_information.head(5)

concept_information.to_csv("aml_concept_information.tsv", sep="\t", index=False)
concept_information.to_excel("aml_concept_information.xlsx", index=False)



In [ ]:
# Save electronic health record concept information 
!gsutil cp aml_concept_information.tsv {my_bucket}/data/phenotype_files/

## CREATE TIME TO EVENT  FILE 

In [ ]:
# Load packages 
import pandas as pd 
import os 

In [ ]:
# =====================================================================================
# Calculate time to event and assign event status 
# =====================================================================================

aml_phenotype_occurrences['entry_datetime'] = (
    pd.to_datetime(aml_phenotype_occurrences['entry_datetime'])
)

earliest_rows = (
    aml_phenotype_occurrences
    .sort_values('entry_datetime') 
    .drop_duplicates(subset='person_id', 
                     keep='first')
)

observation_age_period = pd.read_csv("observation_age_period.tsv", sep ='\t')

earliest_rows = earliest_rows[["person_id", "concept_id", "entry_datetime"]]

earliest_rows["event"] = (
    earliest_rows["entry_datetime"].dt.date
)

time_to_event = earliest_rows.merge(observation_age_period, 
                                    on ="person_id", 
                                    how = "outer")

time_to_event["status"] = (
    time_to_event["event"]
    .apply(lambda x: 1 if pd.isna(x) else 2)
)

time_to_event["event"] = (
    time_to_event["event"]
    .fillna(time_to_event["observation_period_end_date"])
)

time_to_event["observation_period_start_date"] = (
    pd.to_datetime(time_to_event["observation_period_start_date"])
)

time_to_event["event"] = pd.to_datetime(time_to_event["event"])
time_to_event["observation_period_start_date"] = pd.to_datetime(
    time_to_event["observation_period_start_date"]
)

time_to_event["date_of_birth"] = pd.to_datetime(
    time_to_event["date_of_birth"]
)

time_to_event["time_to_event_days_from_observation"] = (
    time_to_event["event"] - time_to_event["observation_period_start_date"]
).dt.days

time_to_event["time_to_event_days_from_birth"] = (
    time_to_event["event"] - time_to_event["date_of_birth"]
).dt.days

time_to_event["age_at_observation_period_start"] = (
    (time_to_event["observation_period_start_date"] - time_to_event["date_of_birth"]
).dt.days / 365.25)

time_to_event["observation_period_start_year"] = (
    time_to_event["observation_period_start_date"].dt.year
)

time_to_event["date_of_birth_year"] = (
    time_to_event["date_of_birth"].dt.year
)

time_to_event.to_csv("aml_phenotype_time_to_event.tsv", 
                     sep ='\t', 
                     index = None)

In [ ]:
# Save the file to workspace bucket  
!gsutil cp aml_phenotype_time_to_event.tsv {my_bucket}/data/covariates/ 